In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..', "scripts"))
# sys.path.insert(0, "C:/Disertation/UoB-GeneTraceAI-25-26/src/scripts")
# Navigate up to 'src' then down to 'scripts'
from pathlib import Path
print(Path.cwd())
print(sys.path)

import pandas as pd
from src.scripts.data_utils import load_raw_parquets, load_clean_parquets

In [ ]:
tables = load_raw_parquets()

In [ ]:
tables_cleaned = load_clean_parquets()

In [ ]:
df_cellosaurus = tables["cellosaurus"]
df_cellosaurus_clean = tables_cleaned["cellosaurus"]

df_cellosaurus_clean.head()

In [ ]:
df_cellosaurus.columns

In [ ]:
df_sample_info = tables["sample_info"]
df_sample_info_cleaned = tables_cleaned["sample_info"]

df_sample_info_cleaned.head()

In [ ]:
df_sample_info.columns

In [ ]:
df_depmap_profiles = tables["depmap_profiles"]
df_depmap_profiles_cleaned = tables_cleaned["depmap_profiles"]


df_depmap_profiles_cleaned.head()

In [ ]:
df_depmap_profiles.columns

## HARMONIZATION

In [ ]:
rrid_valid = df_sample_info['RRID'].str.match(r'^CVCL_', na=False)
print(f"RRID populated: {df_sample_info['RRID'].notna().sum()} / {len(df_sample_info)}")
print(f"RRID well-formed CVCL: {rrid_valid.sum()}")

In [ ]:
direct_match = df_sample_info['RRID'].isin(
        df_cellosaurus['Accession (CVCL_xxxx)']
    )
print(f"Direct RRID -> Accession match: {direct_match.sum()} / {len(df_sample_info)}")

In [ ]:
needs_fallback = df_sample_info[~direct_match]
print(f"Rows needing name fallback: {len(needs_fallback)}")

### sample_info

In [ ]:
# Null value check
assert df_sample_info['DepMap_ID'].isna().sum() == 0

In [ ]:
# Duplicated key check
assert df_sample_info['DepMap_ID'].duplicated().sum() == 0

### cellosaurus

In [ ]:
# Null value check
assert df_cellosaurus['Accession (CVCL_xxxx)'].duplicated().sum() == 0

In [ ]:
# Duplicated key check
assert df_cellosaurus['Accession (CVCL_xxxx)'].isna().sum() == 0

### depmap_profiles

In [ ]:
# Null value check - ModelID
assert df_depmap_profiles["ModelID"].duplicated().sum() == 0

In [ ]:
# Duplicated key check - ModelID
df_depmap_profiles["ModelID"].duplicated().sum()

In [ ]:
# Null value check - ProfileID
assert df_depmap_profiles["ProfileID"].duplicated().sum() == 0

In [ ]:
# Duplicated key check - ModelID
assert df_depmap_profiles["ModelID"].isna().sum() == 0

#### ASSERTING CLEANED COLUMNS 

#### sample_info cleaned

In [ ]:
df_sample_info_cleaned.columns

In [ ]:
# Null value check
assert df_sample_info_cleaned['depmap_id'].isna().sum() == 0

In [ ]:
# Duplicated key check
assert df_sample_info_cleaned['depmap_id'].duplicated().sum() == 0

#### cellosuarus cleaned

In [ ]:
df_cellosaurus_clean.columns

In [ ]:
# Null value check
assert df_cellosaurus_clean['cellosaurus_accession'].duplicated().sum() == 0

In [ ]:
# Duplicated key check
assert df_cellosaurus_clean['cellosaurus_accession'].isna().sum() == 0

#### depmap_profile cleaned

In [ ]:
df_depmap_profiles_cleaned.columns

In [ ]:
# Duplicated value check - ModelID
assert df_depmap_profiles_cleaned["modelid"].duplicated().sum() == 0

In [ ]:
# print the sum of the Duplicated value check - ModelID
print(df_depmap_profiles_cleaned["modelid"].duplicated().sum())

In [ ]:
df_depmap_profiles_cleaned.shape

In [ ]:
# Null key check - ModelID
assert df_depmap_profiles_cleaned["modelid"].isna().sum() == 0

In [ ]:
# Null key check - ModelID
assert df_depmap_profiles_cleaned["profileid"].duplicated().sum() == 0

In [ ]:
df_depmap_profiles_cleaned.loc[
    df_depmap_profiles_cleaned["modelid"].duplicated(keep=False),
]

In [ ]:
# Check the datatype combinations exist per modelid 
df_depmap_profiles_cleaned.groupby("modelid")["datatype"].apply(lambda x: sorted(x.unique())).value_counts()

In [ ]:
# Boolean coverage flags (your existing pivot)
flags_pivoted = (
    df_depmap_profiles_cleaned
    .assign(flag=True)
    .pivot_table(
        index="modelid",
        columns="datatype",
        values="flag",
        aggfunc="any",
        fill_value=False
    )
    .reset_index()
    .rename(columns=lambda c: f"has_{c}" if c != "modelid" else c)
)

flags_pivoted.head()

In [ ]:
# Pivot modelcondition per datatype (one column per datatype)
modelcondition_pivoted = (
    df_depmap_profiles_cleaned
    .pivot_table(
        index="modelid",
        columns="datatype",
        values="modelcondition",
        aggfunc="first"   
    )
    .reset_index()
    .rename(columns=lambda c: f"modelcondition_{c}" if c != "modelid" else c)
)

modelcondition_pivoted.head()

In [ ]:
# Pivot weskit per datatype (only meaningful for 'wes', will be NaN elsewhere)
weskit_pivoted = (
    df_depmap_profiles_cleaned
    .pivot_table(
        index="modelid",
        columns="datatype",
        values="weskit",
        aggfunc="first"
    )
    .reset_index()
    .rename(columns=lambda c: f"weskit_{c}" if c != "modelid" else c)
)

weskit_pivoted.head()

In [ ]:
df_depmap_profiles_cleaned_pivoted = (
    flags_pivoted
    .merge(modelcondition_pivoted, on="modelid", how="left")
    .merge(weskit_pivoted, on="modelid", how="left")
)

assert df_depmap_profiles_cleaned_pivoted["modelid"].duplicated().sum() == 0
print(f"Shape: {df_depmap_profiles_cleaned_pivoted.shape}")

df_depmap_profiles_cleaned_pivoted.head()

In [ ]:
print(f"modelid+datatype duplicate pairs: {df_depmap_profiles_cleaned.duplicated(subset=["modelid", "datatype"]).sum()}")  

In [ ]:
df_depmap_profiles_cleaned.loc[
    df_depmap_profiles_cleaned.duplicated(subset=["modelid", "datatype"])
]

### Key to aggregate the tables:
- depmap_profile: modelid
- cellosaurus: cellosaurus_accession
- sample_info: depmap_id, rrid

### Create the Lookup table 

#### CELLOSAURUS & SAMPLE_INFO HARMONIZATION

#### Check the cellosaurus_accession & rrid map ratio

In [ ]:
df_cellosaurus_clean["cellosaurus_accession"]

In [ ]:
df_sample_info_cleaned["rrid"]

In [ ]:
rrid_valid_id = set(df_sample_info_cleaned["rrid"]) 

print("Mapped cellosaurus_accession & rrid map ratio:", df_cellosaurus_clean[df_cellosaurus_clean["cellosaurus_accession"].isin(rrid_valid_id)]["cellosaurus_accession"].count())
print("Mis-Mapped cellosaurus_accession & rrid map ratio:", df_cellosaurus_clean[~df_cellosaurus_clean["cellosaurus_accession"].isin(rrid_valid_id)]["cellosaurus_accession"].count())

In [ ]:
# --- STEP 1: Normalize both keys ---
cvcl_norm = df_cellosaurus_clean["cellosaurus_accession"].str.strip().str.upper()
rrid_norm  = df_sample_info_cleaned["rrid"].str.strip().str.upper()

cvcl_set = set(cvcl_norm)
rrid_set  = set(rrid_norm)

print("=== After normalization ===")
print("Matched :", len(cvcl_set & rrid_set))
print("rrid NOT in cellosaurus:", len(rrid_set - cvcl_set))

# --- STEP 2: Inspect the 28 unmatched rrid values ---
unmatched = rrid_set - cvcl_set
print("\nUnmatched rrid sample:", list(unmatched)[:15])

# --- STEP 3: Prefix audit ---
print("\nrrid prefix distribution:")
print(df_sample_info_cleaned["rrid"].str[:5].value_counts().head(10))

print("\ncellosaurus_accession prefix distribution:")
print(df_cellosaurus_clean["cellosaurus_accession"].str[:5].value_counts().head(10))

# --- STEP 4: Length distribution (detects padding/extra chars) ---
print("\nrrid length stats:")
print(df_sample_info_cleaned["rrid"].str.len().describe())

print("\ncellosaurus_accession length stats:")
print(df_cellosaurus_clean["cellosaurus_accession"].str.len().describe())

# --- STEP 5: Check for 'RRID:' prefix in rrid column ---
has_rrid_prefix = df_sample_info_cleaned["rrid"].str.startswith("RRID:").sum()
print(f"\nEntries with 'RRID:' prefix in sample_info: {has_rrid_prefix}")

In [ ]:
# Universal normalization before merge
df_cellosaurus_clean["key"] = (df_cellosaurus_clean["cellosaurus_accession"]
                                .str.strip()
                                .str.upper()
                                .str.replace("RRID:", "", regex=False))

df_sample_info_cleaned["key"] = (df_sample_info_cleaned["rrid"]
                                  .str.strip()
                                  .str.upper()
                                  .str.replace("RRID:", "", regex=False))

merged = df_sample_info_cleaned.merge(df_cellosaurus_clean, on="key", how="left")
print("Total merged:", len(merged))
print("Null matches:", merged["cellosaurus_accession"].isna().sum())

## Cellosaurus ↔ RRID Harmonization: Data Loss Summary

1. **Cellosaurus registry** contains **152,231 cell line records**, each identified by a unique `cellosaurus_accession` (format: `cvcl_xxxx`, length = 9, uniform).

2. **Sample info** contains **1,840 experiment-scoped records**, but only **1,818 have a valid `rrid`** remaining **22 are NaN at source**.

3. **Key format audit** both keys share identical format (`cvcl_` prefix, length = 9, no `RRID:` pollution, no case mismatch) → format is **not the cause** of data loss.

4. **Post-normalization match** — `1,812 / 1,818` valid rrid rows matched successfully → only **6 valid-format keys failed** (`CVCL_X507`, `CVCL_V618`, + 4 NaN counted in valid set).

5. **2 accessions (`CVCL_X507`, `CVCL_V618`)** are syntactically valid but **absent from the provided dump** likely secondary/deprecated accessions not captured in the institutional Cellosaurus snapshot.

6. **22 records have null rrid** at source no accession was recorded during data collection; unresolvable without external lookup.

7. **Total irrecoverable data loss = 24 rows (1.30%)** → final harmonized dataset: **1,812 / 1,840 records (98.48% coverage)**.

#### DEPMAP_PROFILE & SAMPLE_INFO HARMONIZATION

#### Check the model_id & depmap_id map ratio

In [ ]:
df_sample_info_cleaned["depmap_id"]

In [ ]:
df_depmap_profiles_cleaned_pivoted["modelid"]

In [ ]:
depmap_id_valid = set(df_sample_info_cleaned["depmap_id"]) 

print("Mapped depmap_id & modelid map ratio:", df_depmap_profiles_cleaned_pivoted[df_depmap_profiles_cleaned_pivoted["modelid"].isin(depmap_id_valid)]["modelid"].count())
print("Mis-Mapped depmap_id & modelid map ratio:", df_depmap_profiles_cleaned_pivoted[~df_depmap_profiles_cleaned_pivoted["modelid"].isin(depmap_id_valid)]["modelid"].count())

In [ ]:
depmap_set  = set(df_sample_info_cleaned["depmap_id"].dropna().str.strip().str.lower())
modelid_set = set(df_depmap_profiles_cleaned_pivoted["modelid"].dropna().str.strip().str.lower())

# Both directions
in_sample_not_profile = depmap_set - modelid_set
in_profile_not_sample = modelid_set - depmap_set

print("sample_info NaN depmap_id     :", df_sample_info_cleaned["depmap_id"].isna().sum())
print("depmap_profiles NaN modelid   :", df_depmap_profiles_cleaned_pivoted["modelid"].isna().sum())
print("depmap_id NOT in modelid      :", len(in_sample_not_profile))
print("modelid NOT in depmap_id      :", len(in_profile_not_sample))

# Format audit
print("\nSample depmap_id prefix   :", df_sample_info_cleaned["depmap_id"].str[:4].value_counts())
print("Sample modelid prefix      :", df_depmap_profiles_cleaned_pivoted["modelid"].str[:4].value_counts())
print("\nSample depmap_id length   :", df_sample_info_cleaned["depmap_id"].str.len().describe())
print("Sample modelid length      :", df_depmap_profiles_cleaned_pivoted["modelid"].str.len().describe())

# Inspect mismatches
print("\nSample unmatched depmap_id :", list(in_sample_not_profile)[:10])
print("Sample unmatched modelid   :", list(in_profile_not_sample)[:10])

In [ ]:
# Inner join → keep only matched records (max valid data)
df_merged = df_sample_info_cleaned.merge(
    df_depmap_profiles_cleaned_pivoted,
    left_on="depmap_id",
    right_on="modelid",
    how="inner"
)

print("Matched rows  :", len(df_merged))           # 1749
print("Lost from sample_info    :", 1840 - len(df_merged))   # 91
print("Lost from depmap_profiles:", 1822 - len(df_merged))   # 73

## DepMap ID ↔ Model ID Harmonization: Data Loss Summary

1. **`sample_info_cleaned`** contains **1,840 records**, each identified by a unique `depmap_id` (format: `ach-XXXXXX`, length = 10, uniform).

2. **`depmap_profiles_cleaned_pivoted`** contains **1,822 records**, identified by `modelid` (same format: `ach-XXXXXX`, length = 10, uniform).

3. **Key format audit** zero nulls, identical prefix (`ach-`), identical length (10) on both sides → format is **not the cause** of data loss.

4. **91 `depmap_id`** records in `sample_info` are absent from `depmap_profiles` cell lines present in metadata but **not profiled** in the provided expression snapshot.

5. **73 `modelid`** records in `depmap_profiles` are absent from `sample_info` profiled cell lines **not captured** in the sample metadata snapshot.

6. Root cause is a **DepMap release version mismatch** between the two datasets both sourced from different snapshot versions, not a data quality issue.

7. **Total irrecoverable data loss = 91 rows (4.95% of sample_info)** → final harmonized dataset after inner join: **1,749 / 1,840 records (95.05% coverage)**.

### OVERALL DATA LOSS ACROSS THE THREE DATASET 
1. **Three datasets in scope** → `cellosaurus_clean` (152,231 rows), `sample_info_cleaned` (1,840 rows), `depmap_profiles_pivoted` (1,822 rows) all keyed on uniform, fixed-length identifiers (`cvcl_` / `ach-`).

2. **Cellosaurus ↔ sample_info** (via `cellosaurus_accession` ↔ `rrid`) → **24 records lost** 22 due to null `rrid` at source, 2 due to secondary/deprecated accessions absent from the provided dump.

3. **sample_info ↔ depmap_profiles** (via `depmap_id` ↔ `modelid`) → **91 records lost** from sample_info side, **73 records lost** from depmap_profiles side caused purely by DepMap release version mismatch between datasets.

4. **Zero format-related losses** across all three joins no case mismatches, no prefix pollution, no null keys in DepMap join, no length inconsistencies detected.

5. **Cumulative loss from sample_info anchor** → `24 (Cellosaurus) + 91 (DepMap) = 115 records` affected across both integrations (overlap unknown without three-way join).

6. **Final harmonized coverage** → Cellosaurus join retains **98.48%** (1,812/1,840); DepMap join retains **95.05%** (1,749/1,840); three-way intersection is the effective working dataset.

7. **Total irrecoverable loss** → **115 records maximum (6.25% of sample_info)** split between source nulls, snapshot version gaps, and deprecated registry entries; none resolvable within the provided data environment.